# 12. CUDA stream ordering around MPI

A walkthrough of [`src/12-stream-aware-mpi.cu`](../src/12-stream-aware-mpi.cu), with the companion [documentation chapter](../docs/source/20-cuda-stream-aware-mpi.rst).

The program fills a GPU buffer, exchanges it with neighbouring MPI ranks, and increments the received values on the GPU. Its central lesson is **finish producing data before MPI reads it, and complete the receive before launching its consumer**.

This notebook explains the source without launching CUDA or MPI. C++ and shell blocks are reading examples. The executable Python cell models the expected values on the CPU; it does not test CUDA/MPI synchronization.

## 1. Execution and ownership

Each MPI process has its own rank, GPU buffers, CUDA streams, and event.

```text
CPU thread                         GPU / communication
launch fill ---------------------> producer stream: fill send_buffer
record ready_for_mpi ------------> event follows fill in that stream
wait for ready_for_mpi <----------- producer's writes complete
MPI_Sendrecv --------------------> send to next rank, receive from previous
MPI_Sendrecv returns <------------ local send and receive complete
launch increment ----------------> consumer stream: increment receive_buffer
wait for consumer <--------------- increment completes
copy first result to CPU
print on rank 0; clean up
```

The order is deliberately sequential for these buffers. Two streams do not by themselves create communication/computation overlap. Blocking MPI stalls the calling CPU thread, not all GPU activity and not necessarily every other rank. `MPI_Sendrecv` is not a global barrier.

## 2. Helper functions and MPI initialization

```cpp
MPI_CHECK(MPI_Init(&argc, &argv));
int rank, size;
MPI_CHECK(MPI_Comm_rank(MPI_COMM_WORLD, &rank));
MPI_CHECK(MPI_Comm_size(MPI_COMM_WORLD, &size));
select_device(MPI_COMM_WORLD);
```

`MPI_COMM_WORLD` contains all participating processes. `rank` identifies this process, from zero to `size - 1`.

[`common.h`](../src/common.h) supplies the error-checking macros and device selection. `CUDA_CHECK` checks CUDA return codes and aborts MPI on failure. `MPI_CHECK` checks MPI return codes if the MPI error handler returns control; MPI's default handler may abort first.

`select_device` constructs a communicator for processes sharing a host, obtains the node-local rank, and selects `local_rank % visible_device_count`. With consistent GPU visibility and one rank per GPU, this assigns distinct devices. More local ranks than visible GPUs causes sharing. Verify placement instead of assuming that a global rank is a valid GPU index.

## 3. Element count and device allocation

```cpp
int count = argc > 1 ? atoi(argv[1]) : 1 << 20;
size_t bytes = (size_t)count * sizeof(double);
double *send_buffer, *receive_buffer;
CUDA_CHECK(cudaMalloc(&send_buffer, bytes));
CUDA_CHECK(cudaMalloc(&receive_buffer, bytes));
```

The optional first argument sets the number of doubles. The default is `2**20 = 1,048,576`, giving 8 MiB per buffer and 16 MiB for both buffers, excluding runtime overhead.

`cudaMalloc` allocates device memory; it does not initialize the elements. `fill` initializes the send buffer and MPI normally fills the receive buffer. These are not pinned host allocations or managed-memory allocations.

Use the same positive count on every rank. This teaching source does not validate `atoi` input: invalid strings, zero, negative values, excessively large counts, or different counts across ranks can cause invalid launches, allocation failures, or message-size errors.

## 4. The two CUDA kernels

```cpp
__global__ void fill(double *buffer, int count, double value)
{
  int index = blockIdx.x * blockDim.x + threadIdx.x;
  if (index < count) {
    buffer[index] = value;
  }
}

__global__ void increment(double *buffer, int count)
{
  int index = blockIdx.x * blockDim.x + threadIdx.x;
  if (index < count) {
    buffer[index] += 1.0;
  }
}
```

`__global__` marks a GPU kernel launched by the CPU. Each thread handles one element. The index combines the block number, threads per block, and thread's position within its block. The bounds check protects excess threads in the final block.

The launch configuration `<<<(count + 255) / 256, 256, 0, stream>>>` means:

| Argument | Meaning |
|---|---|
| `(count + 255) / 256` | Number of blocks, rounding up for a positive count |
| `256` | Threads per block |
| `0` | No dynamic shared memory requested |
| `stream` | CUDA stream in which to enqueue this kernel |

The CPU can continue after enqueueing a kernel, before the GPU finishes it. This is why the producer needs explicit synchronization.

## 5. Streams and the producer event

```cpp
cudaStream_t producer_stream, consumer_stream;
cudaEvent_t ready_for_mpi;
CUDA_CHECK(cudaStreamCreateWithFlags(&producer_stream, cudaStreamNonBlocking));
CUDA_CHECK(cudaStreamCreateWithFlags(&consumer_stream, cudaStreamNonBlocking));
CUDA_CHECK(cudaEventCreateWithFlags(&ready_for_mpi, cudaEventDisableTiming));
```

A stream is an ordered queue of GPU operations. `producer_stream` runs `fill`; `consumer_stream` runs `increment`.

`cudaStreamNonBlocking` means these streams do not implicitly synchronize with the legacy default stream. It does not make MPI nonblocking or guarantee concurrent execution.

`ready_for_mpi` is an event handle, not a Boolean flag. Creating it does not record a point in a stream. `cudaEventDisableTiming` disables timing collection because this event is used only for ordering. Use separate timing-enabled events if measuring GPU durations.

See NVIDIA's [asynchronous execution guide](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/asynchronous-execution.html) and [event API](https://docs.nvidia.com/cuda/cuda-runtime-api/cuda_runtime_api/group__CUDART__EVENT.html).

## 6. Ring neighbours and expected data

```cpp
int peer = (size == 1) ? MPI_PROC_NULL : (rank + 1) % size;
int source = (size == 1) ? MPI_PROC_NULL : (rank + size - 1) % size;
double value = (double)rank;
```

For at least two ranks, each process sends to its successor and receives from its predecessor. Four ranks form `0 → 1 → 2 → 3 → 0`.

Rank `r` fills every send element with `double(r)`. Each received element therefore equals `source`; after `increment`, it equals `source + 1`.

Change `size` in the following CPU-only model to predict a different run. This models values, not communication timing.

In [ ]:
size = 4
assert size >= 2, "The CUDA example needs at least two ranks for initialized receive data."
print("rank  sends_to  receives_from  final_value")
for rank in range(size):
    peer = (rank + 1) % size
    source = (rank + size - 1) % size
    print(f"{rank:4d}  {peer:8d}  {source:13d}  {float(source + 1):11.1f}")

## 7. Why `ready_for_mpi` is necessary

```cpp
fill<<<(count + 255) / 256, 256, 0, producer_stream>>>(
    send_buffer, count, value);
CUDA_CHECK(cudaEventRecord(ready_for_mpi, producer_stream));
CUDA_CHECK(cudaEventSynchronize(ready_for_mpi));
```

The event is recorded after `fill` in the same stream. Its completion therefore establishes completion of the preceding kernel. `cudaEventSynchronize` waits on the CPU until that recorded work has completed. Only then does the CPU call MPI.

Without this dependency, MPI could read `send_buffer` while `fill` is still writing it. CUDA-aware MPI's acceptance of a device pointer does not mean it automatically tracks the application's producer stream.

For this particular example, `cudaStreamSynchronize(producer_stream)` could replace the record-and-wait pair. The event demonstrates waiting for a particular recorded point rather than all work subsequently submitted to a stream. `cudaDeviceSynchronize` would impose a broader device-wide wait.

## 8. The blocking exchange

```cpp
MPI_CHECK(MPI_Sendrecv(
    send_buffer, count, MPI_DOUBLE, peer, 12,
    receive_buffer, count, MPI_DOUBLE, source, 12,
    MPI_COMM_WORLD, MPI_STATUS_IGNORE
));
```

| Arguments | Meaning |
|---|---|
| `send_buffer, count, MPI_DOUBLE` | Send `count` doubles from GPU memory |
| `peer, 12` | Destination and message tag |
| `receive_buffer, count, MPI_DOUBLE` | Receive into a separate GPU buffer |
| `source, 12` | Expected sender and matching tag |
| `MPI_COMM_WORLD` | Communication context |
| `MPI_STATUS_IGNORE` | Discard the returned message status |

`MPI_Sendrecv` combines a send and a receive, avoiding the cyclic wait that can arise when every rank performs a blocking send before posting a receive. All ranks must still use compatible peers, tags, counts, and datatypes.

MPI owns the relevant buffers while the call is in progress: do not change the send data or read/write the receive data concurrently. On return, the local send and receive are complete. Local send completion permits reuse of the send buffer; it does not imply the remote consumer kernel has run.

This requires an MPI build supporting these operations on CUDA device buffers. The implementation may use GPU P2P, GPUDirect RDMA, internal host staging, or another transport. Device pointers alone do not prove which path was used.

## 9. Why there is no `ready_for_kernel`

```cpp
// MPI_Sendrecv has returned here.
increment<<<(count + 255) / 256, 256, 0, consumer_stream>>>(
    receive_buffer, count);
CUDA_CHECK(cudaStreamSynchronize(consumer_stream));
```

For a supported CUDA-aware device-buffer receive, the blocking MPI call establishes receive completion before returning. The CPU launches `increment` only after that return. No extra CUDA event is needed between these two operations.

An event recorded after MPI returns does not track MPI's internal work. A `ready_for_kernel` event and a corresponding stream wait would add no required dependency in this example.

| Boundary | What establishes ordering? |
|---|---|
| `fill` → MPI reads send buffer | CPU waits for `ready_for_mpi` |
| MPI fills receive buffer → `increment` | Blocking `MPI_Sendrecv` returns before kernel launch |
| `increment` → host reads result | CPU waits for `consumer_stream` |

The final stream synchronization serves a different purpose from the removed event: it waits for the consumer kernel itself to finish. It is retained before the host reads the result.

## 10. Reporting and cleanup

```cpp
double sample = 0.0;
CUDA_CHECK(cudaMemcpy(&sample, receive_buffer, sizeof(sample),
                      cudaMemcpyDeviceToHost));
if (!rank) {
  printf("stream-aware MPI: %d ranks, %d values, first received value %.1f\n",
         size, count, sample);
}
```

Each rank copies one double to CPU memory, but only rank 0 prints. Rank 0 receives from `size - 1`, so its final value is `size` for valid runs with at least two ranks.

For two ranks and the default count, the expected program line is:

```text
stream-aware MPI: 2 ranks, 1048576 values, first received value 2.0
```

This is a prediction, not captured GPU test output. Checking one element is a demonstration, not full-array validation.

The code then destroys `ready_for_mpi`, destroys both streams, frees both GPU allocations, and calls `MPI_Finalize`. By this point the blocking exchange and the consumer stream synchronization have completed the work using those resources.

## 11. Building and running on Gadi

Use an allocated compute environment for compilation and execution; do not compile or run GPU programs on a login node. The notebook itself does not submit jobs. Follow the repository's build procedure with the existing CMake target:

```bash
module purge
module load cuda/11.4.1 openmpi/4.1.5 cmake
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=70
cmake --build build --target 12-stream-aware-mpi --parallel
```

These module versions are the repository's configured baseline; check their availability on Gadi. The prebuilt executable is `build/bin/12-stream-aware-mpi`.

The following is a runtime-only PBS script template. Save it as `12-stream-aware-mpi.pbs` in the repository root, after building the executable separately:

```bash
#!/bin/bash
#PBS -P vp91
#PBS -q gpuvolta
#PBS -l ncpus=24
#PBS -l ngpus=2
#PBS -l mem=8gb
#PBS -l jobfs=1GB
#PBS -l walltime=00:10:00
#PBS -l storage=scratch/vp91
#PBS -l wd
#PBS -N stream-mpi
#PBS -j oe

set -euo pipefail
module purge
module load cuda/11.4.1 openmpi/4.1.5

mpirun -np 2 --map-by ppr:2:node:PE=12 --bind-to core --report-bindings \
  ./build/bin/12-stream-aware-mpi 1048576
```

Submit from the repository root with `qsub 12-stream-aware-mpi.pbs`. Request storage matching the actual location of the checkout and executable. For Gadi's job submission conventions, see the [Gadi jobs guide](https://handson-with-gadi.readthedocs.io/en/latest/tutorial/jobs.html).

Inspect the PBS output and rank bindings. Use the repository's `01-rank-device` example in a matching allocation to verify GPU assignment; this stream example does not print device mappings. If allocating a different GPU count, allocate 12 CPU cores per GPU and adjust rank placement together.

## 12. Correctness and portability limits

- **Single rank:** the code uses `MPI_PROC_NULL`. A receive from that source leaves the buffer unchanged, so `increment` reads uninitialized device memory. Use at least two ranks. A future fix could reject single-rank runs or initialize the receive buffer explicitly.
- **Input:** use a positive, reasonably sized element count consistently across ranks. The current source does not validate it.
- **CUDA-aware support:** verify the specific MPI installation and device-buffer operation. A CUDA installation alone does not supply this capability.
- **Kernel errors:** runtime calls are checked, but the source has no explicit immediate launch-error check after each kernel. Synchronization can surface asynchronous execution failures; adding `cudaGetLastError` after launches can improve diagnostics.
- **Validation:** check every received element against `source + 1` when extending the example; the printed sample cannot detect errors elsewhere.
- **Performance:** this is an ordering demonstration, not a bandwidth benchmark. It has no warm-up or repeated timing loop and does not overlap these buffer dependencies.

When timing an extension, use timing-enabled CUDA events for GPU work and `MPI_Wtime` for elapsed MPI/application time. Compare equivalent work and validate results before interpreting performance.

## 13. From this example to nonblocking MPI and Jacobi

Replacing the receive with `MPI_Irecv` changes the completion boundary:

```cpp
MPI_Irecv(receive_buffer, count, MPI_DOUBLE, source, tag,
          MPI_COMM_WORLD, &request);
// Perform independent work here; do not access receive_buffer yet.
MPI_Wait(&request, MPI_STATUS_IGNORE);
increment<<<grid, block, 0, consumer_stream>>>(receive_buffer, count);
```

This fragment assumes a matching send elsewhere. Launching the consumer immediately after `MPI_Irecv` would be incorrect because posting a request does not complete it. `MPI_Wait`, or a successful completion test, must establish completion first. A nonblocking send likewise keeps its send buffer in use until its request completes.

In the Jacobi solver, apply the same reasoning to halo buffers: finish producing outgoing boundaries, post communication, compute independent interior points, complete incoming halo receives, then compute boundary points. Overlap also depends on MPI progress and available GPU/network resources; merely using `MPI_Isend` and `MPI_Irecv` does not guarantee it.

The source demonstrates explicit application ordering around CUDA-aware MPI. It does not call an MPI extension that accepts a CUDA stream, and it does not initiate MPI from GPU threads.

## 14. Exercises

1. Predict rank 0's output for three ranks and `count = 257`. How many CUDA blocks are launched?
2. Explain the race introduced by deleting `cudaEventSynchronize(ready_for_mpi)`. Why is one successful run insufficient evidence of correctness?
3. Replace the producer event with `cudaStreamSynchronize(producer_stream)` in a separate experiment. Which ordering remains the same?
4. Add robust count validation and a defined single-rank policy.
5. Validate every received value on every rank and reduce the number of failures across MPI ranks.
6. Design a nonblocking version with independent GPU work between request posting and completion. Identify which buffers that work may touch.

**Check your reasoning:** for exercise 1, rank 0 prints `3.0` and each kernel launches two blocks. For exercise 2, MPI can read partially produced data; scheduling can conceal the race. For exercise 3, both waits ensure `fill` completes before MPI, while an event identifies a recorded point in its stream.

Return to the [source](../src/12-stream-aware-mpi.cu) and trace each buffer's owner at every synchronization boundary.